In [32]:
import pandas as pd
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

data = pd.read_csv("作业1.csv")
groups = {
    "专家组": ["祁中海", "蹇子欣", "苏艳", "员阳"],
    "学生组": ["革政君", "吾宇旗义轩", "迮静怡", "邴中海"],
    "家长组": ["陈依诺", "出英", "介义轩", "裴丽芳"],
}

data.head()

,图片,真实结果,祁中海,蹇子欣,苏艳,员阳,革政君,吾宇旗义轩,迮静怡,邴中海,陈依诺,出英,介义轩,裴丽芳
0,1,0,1,0,0,0,1,0,0,1,0,0,1,1
1,1,1,0,1,1,1,1,0,0,1,1,0,1,1
2,1,0,0,0,0,0,0,0,1,0,0,0,1,1
3,1,0,0,0,0,0,0,0,1,0,0,0,0,0
4,1,0,0,0,0,0,1,1,0,0,0,0,1,1


In [33]:
records = []
for group, people in groups.items():
    for person in people:
        correct = data[person].eq(data["真实结果"])
        accuracy = correct.groupby(data["图片"]).mean()
        for image_id, value in accuracy.items():
            records.append([person, group, image_id, value])

accuracy_df = pd.DataFrame(records, columns=["人员", "组别", "图片", "判断准确率"])
accuracy_df

,人员,组别,图片,判断准确率
0,祁中海,专家组,1,0.88
1,祁中海,专家组,2,0.82
2,祁中海,专家组,3,0.96
3,祁中海,专家组,4,0.92
4,祁中海,专家组,5,0.88
...,...,...,...,...
67,裴丽芳,家长组,2,0.50
68,裴丽芳,家长组,3,0.58
69,裴丽芳,家长组,4,0.70
70,裴丽芳,家长组,5,0.54


In [34]:
summary = accuracy_df.groupby("组别", sort=False)["判断准确率"].agg(
    观测数="count", 平均准确率="mean", 标准差="std"
)
summary

,观测数,平均准确率,标准差
组别,,,
专家组,24,0.880833,0.043928
学生组,24,0.705833,0.061426
家长组,24,0.575000,0.073248


In [35]:
samples = [
    accuracy_df.loc[accuracy_df["组别"] == group, "判断准确率"]
    for group in groups
]
f_value, p_value = f_oneway(*samples)
df_between = len(groups) - 1
df_within = len(accuracy_df) - len(groups)
print(f"F({df_between}, {df_within}) = {f_value:.6f}，p = {p_value:.6g}")
if p_value < 0.05:
    print("至少有两组的平均判断准确率存在显著差异。")
else:
    print("未发现组间平均判断准确率存在统计学显著差异。")

F(2, 69) = 153.171206，p = 4.19315e-26
至少有两组的平均判断准确率存在显著差异。


In [36]:
if p_value < 0.05:
    tukey = pairwise_tukeyhsd(
        endog=accuracy_df["判断准确率"],
        groups=accuracy_df["组别"],
        alpha=0.05,
    )
    print(tukey)

Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj  lower   upper  reject
---------------------------------------------------
   专家组    学生组   -0.175   0.0  -0.217  -0.133   True
   专家组    家长组  -0.3058   0.0 -0.3478 -0.2638   True
   学生组    家长组  -0.1308   0.0 -0.1728 -0.0888   True
---------------------------------------------------
